# Snowpark Python Notebook — Financial Services Demo
Explore FINSERV_DB tables using Snowpark Python DataFrames, aggregations, VARIANT parsing, and visualizations.

In [ ]:
# Cell 1: Install dependencies
# !pip install snowflake-snowpark-python matplotlib seaborn pandas

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE FINSERV_DB").collect()
session.sql("USE SCHEMA BASE").collect()
session.sql("USE WAREHOUSE FINSERV_WH").collect()
print(f'Connected: {session.get_current_database()}.{session.get_current_schema()}')

## 1. Table Exploration

In [ ]:
for schema in ['BASE', 'RAW', 'CURATED', 'CONSUMPTION']:
    tables = session.sql(f"SHOW TABLES IN SCHEMA FINSERV_DB.{schema}").collect()
    print(f'\n=== {schema} ({len(tables)} tables) ===')
    for t in tables:
        name = t['name']
        count = session.table(f'FINSERV_DB.{schema}.{name}').count()
        print(f'  {name}: {count:,} rows')

In [ ]:
df_customers = session.table('FINSERV_DB.BASE.CUSTOMERS')
df_customers.schema

In [ ]:
# Cell 5: Sample data
df_customers.select('CUSTOMER_ID', 'FIRST_NAME', 'LAST_NAME', 'CITY', 'COUNTRY',
                     'ANNUAL_INCOME', 'CREDIT_SCORE').show(10)

## 2. Aggregations

In [ ]:
# Cell 6: Customer count by country
from snowflake.snowpark.functions import col, count, avg, sum as sum_, round as round_

df_by_country = (
    df_customers
    .group_by('COUNTRY')
    .agg(
        count('*').alias('CUSTOMER_COUNT'),
        round_(avg('ANNUAL_INCOME'), 2).alias('AVG_INCOME'),
        round_(avg('CREDIT_SCORE'), 0).alias('AVG_CREDIT_SCORE')
    )
    .sort(col('CUSTOMER_COUNT').desc())
)
df_by_country.show()

In [ ]:
df_txn = session.table('FINSERV_DB.BASE.TRANSACTIONS')

df_txn_metrics = (
    df_txn
    .group_by('CATEGORY')
    .agg(
        count('*').alias('TXN_COUNT'),
        round_(sum_('AMOUNT'), 2).alias('TOTAL_VOLUME'),
        round_(avg('AMOUNT'), 2).alias('AVG_AMOUNT')
    )
    .sort(col('TOTAL_VOLUME').desc())
)
df_txn_metrics.show()

In [ ]:
# Cell 8: Transaction volume by channel
df_channel = (
    df_txn
    .group_by('CHANNEL')
    .agg(
        count('*').alias('TXN_COUNT'),
        round_(sum_('AMOUNT'), 2).alias('VOLUME'),
        round_(avg('AMOUNT'), 2).alias('AVG_TXN')
    )
    .sort(col('VOLUME').desc())
)
df_channel.show()

## 3. VARIANT / Semi-Structured Data

In [ ]:
df_risk = session.table('FINSERV_DB.BASE.RISK_ASSESSMENTS')

df_risk_parsed = df_risk.select(
    col('CUSTOMER_ID'),
    col('RISK_DATA')['risk_score'].cast('int').alias('RISK_SCORE'),
    col('RISK_DATA')['credit_history'].cast('string').alias('CREDIT_HISTORY'),
    col('RISK_DATA')['debt_to_income'].cast('float').alias('DTI_RATIO'),
    col('RISK_DATA')['assessment_type'].cast('string').alias('ASSESSMENT_TYPE')
)
df_risk_parsed.show(10)

In [ ]:
# Cell 10: Risk score distribution
df_risk_dist = (
    df_risk_parsed
    .group_by('CREDIT_HISTORY')
    .agg(
        count('*').alias('COUNT'),
        round_(avg('RISK_SCORE'), 1).alias('AVG_RISK'),
        round_(avg('DTI_RATIO'), 3).alias('AVG_DTI')
    )
    .sort('CREDIT_HISTORY')
)
df_risk_dist.show()

In [ ]:
df_market = session.table('FINSERV_DB.BASE.MARKET_DATA')

df_market_parsed = df_market.select(
    col('TICKER'),
    col('TRADE_DATE'),
    col('MARKET_DATA')['open'].cast('float').alias('OPEN'),
    col('MARKET_DATA')['high'].cast('float').alias('HIGH'),
    col('MARKET_DATA')['low'].cast('float').alias('LOW'),
    col('MARKET_DATA')['close'].cast('float').alias('CLOSE'),
    col('MARKET_DATA')['volume'].cast('int').alias('VOLUME'),
    col('MARKET_DATA')['indicators']['rsi'].cast('float').alias('RSI')
)
df_market_parsed.sort(col('TRADE_DATE').desc()).show(10)

## 4. Visualizations

In [ ]:
# Cell 12: Setup matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

In [ ]:
# Cell 13: Transaction volume by category (bar chart)
pdf_cat = df_txn_metrics.to_pandas()

fig, ax = plt.subplots()
sns.barplot(data=pdf_cat, x='TOTAL_VOLUME', y='CATEGORY', palette='viridis', ax=ax)
ax.set_title('Transaction Volume by Category')
ax.set_xlabel('Total Volume ($)')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 14: Credit score distribution
pdf_cust = df_customers.select('CREDIT_SCORE', 'ANNUAL_INCOME', 'COUNTRY').to_pandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(pdf_cust['CREDIT_SCORE'], bins=30, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Credit Score Distribution')

sns.histplot(pdf_cust['ANNUAL_INCOME'], bins=30, kde=True, ax=axes[1], color='coral')
axes[1].set_title('Annual Income Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 15: Risk score vs DTI scatter
pdf_risk = df_risk_parsed.to_pandas()

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=pdf_risk, x='DTI_RATIO', y='RISK_SCORE',
                hue='CREDIT_HISTORY', alpha=0.6, ax=ax)
ax.set_title('Risk Score vs Debt-to-Income Ratio')
ax.set_xlabel('Debt-to-Income Ratio')
ax.set_ylabel('Risk Score')
plt.tight_layout()
plt.show()

In [ ]:
print('Notebook complete.')